# satellite-change-agent: ask about a flood, a wildfire or new construction

This notebook runs the measurement of [satellite-change-agent](https://github.com/deltasata/satellite-change-agent)
on Google's computers. **Nothing is installed on your computer**: the code, the trained networks and the
satellite images are all downloaded to Google's temporary machine, which is deleted when you close the tab.

Run the cells from top to bottom with the ▶ buttons, or with *Runtime → Run all*. The first two cells
prepare the machine (about 3-4 minutes); every question after that takes 1-3 minutes, most of it downloading
free Sentinel-2 images.

The language model of the full system (which holds a conversation) is not used here: you give the place,
the kind of event and the date directly, and you get the same measurement the chat agent would call.

## 1. Prepare the machine (about 2 minutes)

In [ ]:
!git clone -q --depth 1 https://github.com/deltasata/satellite-change-agent.git
%cd satellite-change-agent
!pip install -q -e . "omnicloudmask==1.8.1" onnx 2>&1 | grep -v -i "warning\|already" || true

## 2. Get the trained networks (about 1-2 minutes)

The water, radar-water and burn networks come from this project's
[GitHub Release](https://github.com/deltasata/satellite-change-agent/releases/tag/models-v1) (CC BY 4.0), each
checked against its fingerprint. The cloud mask (OmniCloudMask) comes from its authors' own page. The
construction network is not published (its training labels may not be shared), so construction questions
use the simpler change rule here, and the answer says so.

In [ ]:
!python scripts/get_models.py

## 3. Ask a question

**3a.** Choose one of the three examples in the list below. The cell then prints exactly what will be measured.
Or choose **"my own place"** and fill in cell **3b**.

In [ ]:
#@title 3a. Pick an example { run: "auto" }
example = "Conselice, Italy: flood on 23 May 2023"  #@param ["Conselice, Italy: flood on 23 May 2023", "Altadena, California: wildfire in January 2025", "Grünheide, Germany: construction, summer 2019 against summer 2021", "my own place"]

EXAMPLES = {
    "Conselice, Italy: flood on 23 May 2023":
        dict(lat=44.5063, lon=11.8276, size_km=5.0, kind="flood", event_date="2023-05-23"),
    "Altadena, California: wildfire in January 2025":
        dict(lat=34.19, lon=-118.13, size_km=4.0, kind="vegetation_loss", event_date="2025-01-01", event_end="2025-01-31"),
    "Grünheide, Germany: construction, summer 2019 against summer 2021":
        dict(lat=52.395, lon=13.795, size_km=3.0, kind="construction", before_start="2019-06-01", before_end="2019-08-31",
             after_start="2021-06-01", after_end="2021-08-31"),
}


def describe(q):
    """The question in words, with every value that will be used."""
    ns, ew = ("N" if q["lat"] >= 0 else "S"), ("E" if q["lon"] >= 0 else "W")
    lines = [f"Place:  {abs(q['lat']):.4f} {ns}, {abs(q['lon']):.4f} {ew}; a square of {q['size_km']:g} km x {q['size_km']:g} km",
             "Event:  " + {"flood": "flood", "vegetation_loss": "wildfire (burnt vegetation)", "construction": "new construction"}[q["kind"]]]
    if q["kind"] == "construction":
        lines.append(f"Compare: {q['before_start']} to {q['before_end']}  against  {q['after_start']} to {q['after_end']}")
    elif q.get("event_end"):
        lines.append(f"Dates:  {q['event_date']} to {q['event_end']} (the images before and after are chosen automatically)")
    else:
        lines.append(f"Date:   {q['event_date']} (the images before and after are chosen automatically)")
    return "\n".join(lines)


question = EXAMPLES.get(example)
if question:
    print("This will be measured:\n" + describe(question))
    print("\n(The boxes of cell 3b are not used while an example is chosen.)")
else:
    print('"my own place": fill in cell 3b and run it.')

**3b. My own place** (used only when 3a says "my own place"). Each box says when it counts:
- **flood**: the day of the flood;
- **wildfire**: the day of the fire; if you only know the month, its first day here and its last day in `fire_last_day`;
- **construction**: two periods to compare (no event day), ideally the same months of two different years.

Find latitude and longitude by right-clicking a place in Google Maps (north and east are positive).

In [ ]:
#@title 3b. My own place { run: "auto" }
latitude = 0.0  #@param {type:"number"}
longitude = 0.0  #@param {type:"number"}
kind = "flood"  #@param ["flood", "wildfire", "construction"]
event_day_flood_or_fire = ""  #@param {type:"string"}
fire_last_day = ""  #@param {type:"string"}
construction_before = ""  #@param {type:"string"}
construction_after = ""  #@param {type:"string"}
size_km = 5.0  #@param {type:"number"}
#@markdown Dates as YYYY-MM-DD; a construction period as `YYYY-MM-DD to YYYY-MM-DD`.

if example != "my own place":
    print(f'Not used: cell 3a has the example "{example}" chosen.')
else:
    question = None
    k = {"flood": "flood", "wildfire": "vegetation_loss", "construction": "construction"}[kind]
    q = dict(lat=latitude, lon=longitude, kind=k, size_km=size_km)
    problem = None
    if latitude == 0 and longitude == 0:
        problem = "Please give the latitude and longitude of your place."
    elif k == "construction":
        try:
            (b0, b1), (a0, a1) = ([x.strip() for x in p.split(" to ")] for p in (construction_before, construction_after))
            q.update(before_start=b0, before_end=b1, after_start=a0, after_end=a1)
        except ValueError:
            problem = "For construction, give both periods as YYYY-MM-DD to YYYY-MM-DD."
    elif not event_day_flood_or_fire.strip():
        problem = "Please give the day of the event (YYYY-MM-DD)."
    else:
        q["event_date"] = event_day_flood_or_fire.strip()
        if k == "vegetation_loss" and fire_last_day.strip():
            q["event_end"] = fire_last_day.strip()
    if problem:
        print(problem)
    else:
        question = q
        print("This will be measured:\n" + describe(question))
        ignored = [n for n, v, used in (("fire_last_day", fire_last_day, k == "vegetation_loss"),
                                        ("construction_before", construction_before, k == "construction"),
                                        ("construction_after", construction_after, k == "construction"),
                                        ("event_day_flood_or_fire", event_day_flood_or_fire, k != "construction"))
                   if v.strip() and not used]
        if ignored:
            print(f"\n(Not used for a {kind} question: {', '.join(ignored)}.)")

In [ ]:
#@title 4. Measure (1-3 minutes)
import warnings
warnings.filterwarnings("ignore")
from IPython.display import Image, display
from core.measure import measure_change

if not question:
    print("No complete question yet: see cell 3a or 3b.")
else:
    print("Measuring:\n" + describe(question) + "\n")
    r = measure_change(**question)
    print("ANSWER:", r.answer.upper().replace("_", " "))
    print("rule:  ", r.rule or r.reason)
    if r.before_period and r.after_period:
        print(f"periods used: before {r.before_period[0]} to {r.before_period[1]}, after {r.after_period[0]} to {r.after_period[1]}")
    if r.caution:
        print("caution:", r.caution)
    print()
    for s in r.steps:
        print(" -", s)
    if r.figure:
        display(Image(r.figure))

### What you see

- **The answer** comes from a fixed rule on the measured numbers (written out on the "rule" line): for a flood, the
  share of the area that is water after the event but not before must be at least 5 %; for a wildfire or new
  construction, the share of the area that changed must be at least 3 %.
- **The figure**: the before and after images (each the median of its cloud-free views), the pixels the network
  marked (blue: new water; orange: burnt or changed), and the cloud-free share of every satellite pass, with the two
  compared periods shaded.
- **Steps**: how many images were found, how much of the ground was seen clearly, and which networks measured.

How each network was chosen, and the tests it had to pass before it was used:
[docs/METHODS.md](https://github.com/deltasata/satellite-change-agent/blob/main/docs/METHODS.md).
Satellite data: Copernicus Sentinel-2 (ESA), via Element 84 Earth Search; radar: Sentinel-1 via Microsoft Planetary Computer.